In [2]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
from datetime import datetime

In [5]:
class MatchDataCollector:
    def __init__(self, root):
        self.root = root
        self.root.title("Tennis Match Data Collector")

        # Match-wide metadata (constant across points)
        self.metadata = {}
        self.point_data = []
        self.point_counter = 1

        # Changing point-by-point data
        self.Set1 = 0
        self.Set2 = 0
        self.Gm1 = 0
        self.Gm2 = 0
        self.Svr = 1  # Player 1 serves first

        self.build_metadata_screen()

    def clear_screen(self):
        for widget in self.root.winfo_children():
            widget.destroy()

    def build_metadata_screen(self):
        self.clear_screen()

        tk.Label(self.root, text="Player 1 (lowercase):").pack()
        self.p1_entry = tk.Entry(self.root)
        self.p1_entry.pack()

        tk.Label(self.root, text="Player 2 (lowercase):").pack()
        self.p2_entry = tk.Entry(self.root)
        self.p2_entry.pack()

        tk.Label(self.root, text="Player 1 Hand (L/R):").pack()
        self.p1_hand_entry = tk.Entry(self.root)
        self.p1_hand_entry.pack()

        tk.Label(self.root, text="Player 2 Hand (L/R):").pack()
        self.p2_hand_entry = tk.Entry(self.root)
        self.p2_hand_entry.pack()

        tk.Label(self.root, text="Surface (hard/clay/grass):").pack()
        self.surface_entry = tk.Entry(self.root)
        self.surface_entry.pack()

        tk.Label(self.root, text="Match Date (YYYYMMDD):").pack()
        self.date_entry = tk.Entry(self.root)
        self.date_entry.pack()

        tk.Button(self.root, text="Start Match", command=self.store_metadata).pack(pady=10)

    def store_metadata(self):
        try:
            datetime.strptime(self.date_entry.get().strip(), "%Y%m%d")
        except ValueError:
            messagebox.showerror("Invalid Date", "Date must be in YYYYMMDD format.")
            return

        self.metadata = {
            "Player1": self.p1_entry.get().strip(),
            "Player2": self.p2_entry.get().strip(),
            "P1_hand": self.p1_hand_entry.get().strip().upper(),
            "P2_hand": self.p2_hand_entry.get().strip().upper(),
            "Surface": self.surface_entry.get().strip().lower(),
            "date": self.date_entry.get().strip(),
        }
        self.build_point_input_screen()

    def build_point_input_screen(self):
        self.clear_screen()

        self.set_display = tk.Label(self.root, text=f"Set Score: {self.Set1}-{self.Set2} | Game Score: {self.Gm1}-{self.Gm2}")
        self.set_display.pack()

        self.server_display = tk.Label(self.root, text=f"Current Server: Player {self.Svr}")
        self.server_display.pack()

        tk.Button(self.root, text="Toggle Server", command=self.toggle_server).pack()

        tk.Button(self.root, text="Player 1 Wins Point", command=lambda: self.record_point(1)).pack(pady=5)
        tk.Button(self.root, text="Player 2 Wins Point", command=lambda: self.record_point(2)).pack(pady=5)

        tk.Button(self.root, text="Player 1 Wins Game", command=lambda: self.update_games(1)).pack()
        tk.Button(self.root, text="Player 2 Wins Game", command=lambda: self.update_games(2)).pack()

        tk.Button(self.root, text="Player 1 Wins Set", command=lambda: self.update_sets(1)).pack()
        tk.Button(self.root, text="Player 2 Wins Set", command=lambda: self.update_sets(2)).pack()

        tk.Button(self.root, text="Finish Match", command=self.finish_match).pack(pady=10)

    def toggle_server(self):
        self.Svr = 2 if self.Svr == 1 else 1
        self.server_display.config(text=f"Current Server: Player {self.Svr}")

    def update_games(self, winner):
        if winner == 1:
            self.Gm1 += 1
        else:
            self.Gm2 += 1
        self.set_display.config(text=f"Set Score: {self.Set1}-{self.Set2} | Game Score: {self.Gm1}-{self.Gm2}")

    def update_sets(self, winner):
        if winner == 1:
            self.Set1 += 1
        else:
            self.Set2 += 1
        self.Gm1 = 0
        self.Gm2 = 0
        self.set_display.config(text=f"Set Score: {self.Set1}-{self.Set2} | Game Score: {self.Gm1}-{self.Gm2}")

    def record_point(self, pt_winner):
        row = [
            self.metadata["Player1"],
            self.metadata["Player2"],
            self.metadata["P1_hand"],
            self.metadata["P2_hand"],
            self.metadata["Surface"],
            "",  # match_winner (filled at end)
            self.metadata["date"],
            self.point_counter,
            self.Set1,
            self.Set2,
            self.Gm1,
            self.Gm2,
            self.Svr,
            pt_winner
        ]
        self.point_data.append(row)
        self.point_counter += 1

    def finish_match(self):
        if self.Set1 > self.Set2:
            winner = 1
        elif self.Set2 > self.Set1:
            winner = 2
        else:
            messagebox.showerror("Error", "Set score is tied. Cannot determine match winner.")
            return

        for row in self.point_data:
            row[5] = winner  # Fill in match_winner

        df = pd.DataFrame(self.point_data, columns=[
            "Player1", "Player2", "P1_hand", "P2_hand", "Surface",
            "match_winner", "date", "Pt", "Set1", "Set2", "Gm1", "Gm2", "Svr", "PtWinner"
        ])

        filename = f"{self.metadata['Player1']}_vs_{self.metadata['Player2']}_{self.metadata['date']}.csv"
        df.to_csv(filename, index=False)
        messagebox.showinfo("Match Saved", f"Match data saved to {filename}")
        self.root.quit()


# Launch the GUI
if __name__ == "__main__":
    root = tk.Tk()
    app = MatchDataCollector(root)
    root.geometry("400x600")
    root.resizable(True, True)
    root.mainloop()
